In [2]:
import os
import polars as pl

# 설정 변수 (기존 유지)
DATA_DIR = "dataset"
JOINED_PATH = os.path.join(DATA_DIR, 'joined', 'joined_all_users.csv')
users_path = os.path.join(DATA_DIR, 'users.csv')

join_files = [
    os.path.join(DATA_DIR, 'events.csv'),
    os.path.join(DATA_DIR, 'orders.csv'),
    os.path.join(DATA_DIR, 'order_items.csv')
]

# 디렉토리 생성
os.makedirs(os.path.dirname(JOINED_PATH), exist_ok=True)

# 1. Users 데이터 LazyFrame으로 로드
# scan_csv를 사용하여 메타데이터만 먼저 읽습니다.
print(f"Scanning users: {users_path}")
users_lf = pl.scan_csv(users_path)

# id -> user_id 컬럼명 통일 및 타입 캐스팅 (String으로 통일하는 것이 안전)
if 'user_id' not in users_lf.collect_schema().names() and 'id' in users_lf.collect_schema().names():
    users_lf = users_lf.rename({'id': 'user_id'})

# Join Key 타입 강제 변환 (Lazy 모드에서는 스키마가 중요)
users_lf = users_lf.with_columns(pl.col('user_id').cast(pl.Utf8))

collected_lfs = []

# 2. 각 파일을 순회하며 Lazy 연산 정의
for p in join_files:
    if not os.path.exists(p):
        print(f"[Warning] File not found, skipping: {p}")
        continue

    name = os.path.basename(p).replace('.csv', '')
    print(f'Preparing plan for: {name}')
    
    # Lazy 로드
    lf = pl.scan_csv(p)
    
    # user_id가 있는 경우에만 처리 (혹은 로직에 따라 없어도 처리할지 결정 필요)
    if 'user_id' in lf.collect_schema().names():
        lf = lf.with_columns(pl.col('user_id').cast(pl.Utf8))
        
        # Left Join 수행
        # 사용자 정보가 없더라도 원본 데이터(events 등)는 유지되어야 하므로 Left Join
        joined = lf.join(users_lf, on='user_id', how='left')
    else:
        # user_id가 없는 파일이라면 join 없이 원본만 사용하거나, 로직에 따라 제외
        print(f"  - 'user_id' column missing in {name}. Skipping join with users.")
        joined = lf

    # Source 컬럼 추가
    joined = joined.with_columns(pl.lit(name).alias('source'))
    
    collected_lfs.append(joined)


# 3. Diagonal Concat (대각선 병합)
# 컬럼이 서로 달라도 자동으로 합집합을 만들고 없는 값은 null로 채움 (기존 수동 로직 대체)
print("Concatenating LazyFrames...")
combined_lf = pl.concat(collected_lfs, how='diagonal')

# 4. Streaming 저장 (sink_csv)
# collect()를 호출하면 메모리에 다 올리게 되므로, sink_csv로 스트리밍 저장합니다.
print(f"Streaming result to: {JOINED_PATH}")
combined_lf.sink_csv(JOINED_PATH)
print("Done.")

# 결과 확인 (처음 5줄만 읽기)
print("\n--- Result Preview ---")
print(pl.read_csv(JOINED_PATH, n_rows=5))


Scanning users: dataset/users.csv
Preparing plan for: events
Preparing plan for: orders
Preparing plan for: order_items
Concatenating LazyFrames...
Streaming result to: dataset/joined/joined_all_users.csv
Done.

--- Result Preview ---
shape: (5, 38)
┌─────────┬─────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ id      ┆ user_id ┆ sequence_n ┆ session_id ┆ … ┆ gender_rig ┆ product_i ┆ inventory ┆ sale_pric │
│ ---     ┆ ---     ┆ umber      ┆ ---        ┆   ┆ ht         ┆ d         ┆ _item_id  ┆ e         │
│ i64     ┆ str     ┆ ---        ┆ str        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│         ┆         ┆ i64        ┆            ┆   ┆ str        ┆ str       ┆ str       ┆ str       │
╞═════════╪═════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2198523 ┆ null    ┆ 3          ┆ 83889ed2-2 ┆ … ┆ null       ┆ null      ┆ null      ┆ null      │
│         ┆         ┆            ┆ adc-4b9a

In [3]:
# Product-based join: merge product-related files with `products.csv`
# - Left join so product info is kept when available; if product not found, product columns remain null
# - Prefer polars LazyFrames and diagonal concat for different schemas; fallback to pandas chunked merge

PRODUCT_JOINED_PATH = os.path.join(DATA_DIR, 'joined', 'joined_all_products.csv')
os.makedirs(os.path.dirname(PRODUCT_JOINED_PATH), exist_ok=True)

product_files = [
    os.path.join(DATA_DIR, 'inventory_items.csv'),
    os.path.join(DATA_DIR, 'order_items.csv')
]

print('Scanning products')
products_lf = pl.scan_csv(os.path.join(DATA_DIR, 'products.csv'))
# rename id->product_id for consistent join key
if 'product_id' not in products_lf.collect_schema().names() and 'id' in products_lf.collect_schema().names():
    products_lf = products_lf.rename({'id': 'product_id'})
products_lf = products_lf.with_columns(pl.col('product_id').cast(pl.Utf8))

collected_pls = []
for p in product_files:
    if not os.path.exists(p):
        print(f"Skipping missing file: {p}")
        continue
    name = os.path.basename(p).replace('.csv','')
    print('Preparing product plan for', name)
    lf = pl.scan_csv(p)
    if 'product_id' in lf.collect_schema().names():
        lf = lf.with_columns(pl.col('product_id').cast(pl.Utf8))
        joined = lf.join(products_lf, on='product_id', how='left')
    else:
        # For files that don't have product_id, just include as-is
        print('  - No product_id in', name, '- including raw rows')
        joined = lf
    joined = joined.with_columns(pl.lit(name).alias('source'))
    collected_pls.append(joined)

print('Concatenating product frames...')
combined_products_lf = pl.concat(collected_pls, how='diagonal')
print('Streaming product join to', PRODUCT_JOINED_PATH)
combined_products_lf.sink_csv(PRODUCT_JOINED_PATH)
print('Product join written')

# preview
print('\n--- Product join preview ---')
print(pl.read_csv(PRODUCT_JOINED_PATH, n_rows=5))


Scanning products
Preparing product plan for inventory_items
Preparing product plan for order_items
Concatenating product frames...
Streaming product join to dataset/joined/joined_all_products.csv
Product join written

--- Product join preview ---
shape: (5, 29)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id     ┆ product_id ┆ created_at ┆ sold_at   ┆ … ┆ shipped_a ┆ delivered ┆ returned_ ┆ sale_pric │
│ ---    ┆ ---        ┆ ---        ┆ ---       ┆   ┆ t         ┆ _at       ┆ at        ┆ e         │
│ i64    ┆ i64        ┆ str        ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│        ┆            ┆            ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 148572 ┆ 13682      ┆ 2023-05-19 ┆ 2023-05-2 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│        ┆            ┆ 08:47:

In [8]:
import os
import polars as pl

# ... (이전 경로 설정 및 Lazy Loading 코드는 동일) ...
DATA_DIR = "dataset"
JOINED_SOURCE_PATH = os.path.join(DATA_DIR, 'joined', 'joined_all_users.csv') 
USER_ORDER_COUNTS_CSV = os.path.join(DATA_DIR, 'joined', 'joined_user_order_counts.csv')

print(f'Scanning combined dataset: {JOINED_SOURCE_PATH}')

# 1. 원본 데이터 Scan
lf_main = pl.scan_csv(JOINED_SOURCE_PATH)
if 'user_id' in lf_main.collect_schema().names():
    lf_main = lf_main.with_columns(pl.col('user_id').cast(pl.Utf8))

# 2. 구매 횟수 집계 (Lookup Table)
lf_stats = (
    lf_main
    .select(['user_id', 'order_id'])
    .filter(
        pl.col('user_id').is_not_null() & 
        pl.col('order_id').is_not_null()
    )
    .group_by('user_id')
    .agg(
        pl.col('order_id').n_unique().alias('order_count')
    )
)

# 3. Left Join
lf_joined = lf_main.join(lf_stats, on='user_id', how='left')

# 4. 파생 변수 생성 및 결측치 처리 [이 부분이 수정됨]
lf_final = lf_joined.with_columns(
    pl.col('order_count').fill_null(0)
).with_columns(
    # [핵심 수정] cast(pl.Int64)를 먼저 수행하여 Underflow 방지
    (pl.col('order_count').cast(pl.Int64) - 1).clip(lower_bound=0).alias('repurchase_count'),
    (pl.col('order_count') > 1).alias('has_repurchase')
)

# 5. 저장
print(f"Streaming result to: {USER_ORDER_COUNTS_CSV}")
lf_final.sink_csv(USER_ORDER_COUNTS_CSV)
print('Done.')

# 확인용 (비회원 케이스 필터링하여 확인)
print("\n--- Check for Underflow (Non-member/No-order rows) ---")
# repurchase_count가 비정상적으로 큰 값이 있는지 확인
print(
    pl.read_csv(USER_ORDER_COUNTS_CSV)
    .filter(pl.col('repurchase_count') > 10000)
    .head()
)

Scanning combined dataset: dataset/joined/joined_all_users.csv
Streaming result to: dataset/joined/joined_user_order_counts.csv
Done.

--- Check for Underflow (Non-member/No-order rows) ---
Done.

--- Check for Underflow (Non-member/No-order rows) ---
shape: (0, 41)
┌─────┬─────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id  ┆ user_id ┆ sequence_nu ┆ session_id ┆ … ┆ sale_price ┆ order_coun ┆ repurchase ┆ has_repurc │
│ --- ┆ ---     ┆ mber        ┆ ---        ┆   ┆ ---        ┆ t          ┆ _count     ┆ hase       │
│ i64 ┆ f64     ┆ ---         ┆ str        ┆   ┆ str        ┆ ---        ┆ ---        ┆ ---        │
│     ┆         ┆ i64         ┆            ┆   ┆            ┆ i64        ┆ i64        ┆ bool       │
╞═════╪═════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
└─────┴─────────┴─────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘
shape: (0, 41)
┌─────┬────